# Photographic stress tests with paired predictions

The notebook records the experiment configuration, analysis procedure and saved outputs for this study.


## 1. Configuration and durable monitoring


In [ ]:
from pathlib import Path
import os, io, json, hashlib, threading, time, sys, contextlib, shutil, traceback
from datetime import datetime, timezone
from urllib.parse import urlparse
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

V4=Path(os.getenv('DATEPALM_V4_ROOT','external_data')).expanduser().resolve()
STAGE=Path(os.getenv('DATEPALM_V51_ROOT','external_data')).expanduser().resolve()
SPATIAL=STAGE/'v51_locked_spatial'
PILOT=STAGE/'v51_grader_decision_pilot'
RGB=STAGE/'v51_representation_finetune_pilot'
TARGETED=STAGE/'v51_dandhi_targeted_rgb_pilot'
AUDIT=STAGE/'v51_candidate_evidence_audit'
OUT=Path(os.getenv('DATEPALM_STRESS_V4_OUT',str(STAGE/'additional_evidence_20261024'/'development_stress_resumable_v4'))).expanduser().resolve()
RUN_INFERENCE=True
REQUIRE_GPU=True               # Avoid unintended hours-long CPU inference
BATCH_SIZE='auto'              # Benchmark best throughput on available GPU
MAX_AUTO_BATCH=512             # Change only for resource limits
NUM_WORKERS='auto'             # Match CPU affinity
AUTO_MIXED_PRECISION=True      # FP16 used only if its pilot matches FP32
AMP_MAX_PROB_DIFF=0.002        # Maximum allowed pilot probability difference
HEARTBEAT_SECONDS=15
VERIFY_ORIGINAL_HASHES=True    # Required for first use, cheap on 835 images
SEEDS=(11,23,37)
SIZES=('Small','Medium','Large')
CONDITIONS=('clean','jpeg_35','low_res_96','blur_1_2','dark_0_7','bright_1_3','contrast_0_7')
EXPERIMENT_VERSION='datepalm-development-stress-v4-20261024'

# The V5.1 STAGE is a *read-only source root* for this experiment, except for
# its specifically allocated additional_evidence_20261024 output namespace.
# OUT is intentionally below STAGE: treating every ancestor as forbidden would
# incorrectly reject the default output path before the first image is loaded.
SAFE_EVIDENCE_ROOT=(STAGE/'additional_evidence_20261024').resolve()
def _same_or_inside(child, parent):
    return child == parent or parent in child.parents

# Reject writing into any frozen data/model root, or a parent that could
# contain it. Path.resolve() (used above) also detects symlink redirection.
for protected in (V4, SPATIAL, PILOT, RGB, TARGETED, AUDIT):
    if _same_or_inside(OUT, protected) or _same_or_inside(protected, OUT):
        raise ValueError(f'Unsafe output overlaps protected input: output={OUT}; protected={protected}')

# If the output is inside STAGE, it MUST lie below the reserved evidence root.
# If it is outside STAGE, it must not contain STAGE (or any protected path).
if _same_or_inside(OUT, STAGE):
    if OUT == SAFE_EVIDENCE_ROOT or not _same_or_inside(OUT, SAFE_EVIDENCE_ROOT):
        raise ValueError(f'Output within STAGE must be a dedicated subfolder of {SAFE_EVIDENCE_ROOT}: {OUT}')
elif _same_or_inside(STAGE, OUT):
    raise ValueError(f'Output cannot be a parent of the frozen staging tree: {OUT}')

OUT.mkdir(parents=True,exist_ok=True)
LOGFILE=OUT/'execution_events.jsonl'
_LOG_LOCK=threading.Lock()
def utc():return datetime.now(timezone.utc).isoformat()
def event(stage,message,**extra):
    record={'utc':utc(),'stage':stage,'message':str(message),**extra}
    with _LOG_LOCK:
        with LOGFILE.open('a',encoding='utf8') as f:
            f.write(json.dumps(record,default=str,sort_keys=True)+'\n');f.flush();os.fsync(f.fileno())
    print(f"[{record['utc'][11:19]}] {stage} | {message}",flush=True)
def atomic_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp')
    with temp.open('w',encoding='utf8') as f:
        json.dump(obj,f,sort_keys=True,indent=2,default=str);f.flush();os.fsync(f.fileno())
    os.replace(temp,path)
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''):h.update(b)
    return h.hexdigest()
def fingerprint(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

STATUS={'phase':'initialized','completed_batches':0,'total_batches':0,'last_progress':time.monotonic()}
class Heartbeat:
    def __enter__(self):
        self.done=threading.Event()
        def runner():
            while not self.done.wait(HEARTBEAT_SECONDS):
                phase=STATUS['phase'];elapsed=int(time.monotonic()-STATUS['last_progress'])
                msg=f"heartbeat | phase={phase} | {STATUS['completed_batches']}/{STATUS['total_batches']} batches | seconds since last progress={elapsed}"
                event('HEARTBEAT',msg)
        self.worker=threading.Thread(target=runner,daemon=True)
        self.worker.start();return self
    def __exit__(self,*args):self.done.set();self.worker.join(timeout=2)

def progress(phase,done=None,total=None):
    STATUS['phase']=phase;STATUS['last_progress']=time.monotonic()
    if done is not None:STATUS['completed_batches']=int(done)
    if total is not None:STATUS['total_batches']=int(total)

print('Output:',OUT,'\nRun inference:',RUN_INFERENCE,'\nGPU required:',REQUIRE_GPU,'\nHeartbeat:',HEARTBEAT_SECONDS,'s',flush=True)
event('START','Resumable development stress notebook initialised')

## 2. Source-verified manifest and verified input images


In [ ]:
mfile=V4/'data/source_split.csv'
pfile=STAGE/'verified_v4_image_path_map.csv'
assert mfile.is_file() and pfile.is_file(), f'Missing manifest/map: {mfile}; {pfile}'
M=pd.read_csv(mfile,dtype=str,keep_default_na=False)
F=pd.read_csv(pfile,dtype=str,keep_default_na=False)
assert M.source_id.is_unique and F.source_id.is_unique
assert M.groupby('source_group').split.nunique().max()==1, 'Source group crosses splits'
assert M.groupby('file_sha256').split.nunique().max()==1, 'Identical bytes cross splits'
M=M[M.eligibility=='eligible'].copy()
assert M.split.value_counts().to_dict()=={'train':3603,'development':835,'locked_test':834,'calibration':556}
DEV=M[M.split=='development'].merge(F[['source_id','verified_image_path','file_sha256']],on='source_id',validate='one_to_one',suffixes=('','_map'))
DEV=DEV.sort_values('source_id').reset_index(drop=True)
assert len(DEV)==835 and DEV.source_group.nunique()==835
assert DEV.file_sha256.str.lower().eq(DEV.file_sha256_map.str.lower()).all()
assert DEV.native_size_label.isin(SIZES).all()
assert DEV.variety.isin(('Aseel','Dandhi','Karblian','Kupro')).all()
assert DEV.verified_image_path.map(lambda p:Path(p).is_file()).all()
if VERIFY_ORIGINAL_HASHES:
    progress('verifying original image hashes')
    with Heartbeat():
        for r in tqdm(DEV.itertuples(index=False),total=len(DEV),desc='Verify development images'):
            got=file_sha(r.verified_image_path)
            assert got.lower()==r.file_sha256.lower(),f'Image hash mismatch: {r.source_id}'
            progress('verifying original image hashes')
    event('INPUT','All 835 development originals SHA-256 verified')
else:
    raise RuntimeError('Original image hash verification must remain enabled for cache integrity')
INPUT_ID=fingerprint({'version':EXPERIMENT_VERSION,'source_ids':DEV.source_id.tolist(),
 'groups':DEV.source_group.tolist(),'sha':DEV.file_sha256.str.lower().tolist(),
 'sizes':DEV.native_size_label.tolist(),'varieties':DEV.variety.tolist(),
 'conditions':CONDITIONS,'input_image_transform':'exif-rgb-bicubic-thumb256-grey128;bilinear224;imagenet'})
atomic_json(OUT/'input_identity.json',{'input_fingerprint':INPUT_ID,'count':len(DEV),'development_only':True})
print('Input identity:',INPUT_ID[:20]);display(DEV.groupby(['variety','native_size_label']).size().unstack(fill_value=0))

## 3. Cache canonical 256×256 images only once


In [ ]:
from PIL import Image,ImageOps,ImageEnhance,ImageFilter
from torchvision import transforms,models
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import accuracy_score,f1_score

PRE=transforms.Compose([transforms.Resize((224,224),interpolation=transforms.InterpolationMode.BILINEAR),
 transforms.ToTensor(),transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
def canonical_image(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((256,256),Image.Resampling.BICUBIC)
        bg=Image.new('RGB',(256,256),(128,128,128))
        bg.paste(im,((256-im.width)//2,(256-im.height)//2))
    return bg

def apply_stress(image,kind):
    if kind=='clean':return image
    if kind=='jpeg_35':
        f=io.BytesIO();image.save(f,format='JPEG',quality=35);f.seek(0)
        with Image.open(f) as x:return x.convert('RGB').copy()
    if kind=='low_res_96':return image.resize((96,96),Image.Resampling.BILINEAR).resize((256,256),Image.Resampling.BICUBIC)
    if kind=='blur_1_2':return image.filter(ImageFilter.GaussianBlur(radius=1.2))
    if kind=='dark_0_7':return ImageEnhance.Brightness(image).enhance(.7)
    if kind=='bright_1_3':return ImageEnhance.Brightness(image).enhance(1.3)
    if kind=='contrast_0_7':return ImageEnhance.Contrast(image).enhance(.7)
    raise ValueError(kind)

CANON_DIR=OUT/'canonical_cache';CANON_DIR.mkdir(exist_ok=True)
CANON_FINAL=CANON_DIR/f'{INPUT_ID[:24]}.npy'
CANON_TEMP=CANON_DIR/f'{INPUT_ID[:24]}.partial.npy'
CANON_STATE=CANON_DIR/f'{INPUT_ID[:24]}.state.json'
CANON_META=CANON_DIR/f'{INPUT_ID[:24]}.complete.json'
SHAPE=(len(DEV),256,256,3)
if CANON_FINAL.exists():
    if not CANON_META.exists() and CANON_STATE.exists():
        recovery=json.loads(CANON_STATE.read_text())
        assert recovery['input_fingerprint']==INPUT_ID and int(recovery['committed'])==len(DEV), 'Incomplete cache cannot be promoted'
        assert np.load(CANON_FINAL,mmap_mode='r').shape==SHAPE
        atomic_json(CANON_META,{'input_fingerprint':INPUT_ID,'sha256':file_sha(CANON_FINAL),
                    'shape':SHAPE,'recovered_utc':utc()})
        CANON_STATE.unlink(missing_ok=True)
        event('RESUME','Recovered complete canonical cache metadata after interrupted finalization')
    assert CANON_META.exists(),'Final cache without identity manifest'
    meta=json.loads(CANON_META.read_text())
    assert meta['input_fingerprint']==INPUT_ID and meta['sha256']==file_sha(CANON_FINAL), 'Canonical cache changed'
    assert np.load(CANON_FINAL,mmap_mode='r').shape==SHAPE
    event('CACHE','Reusing complete verified canonical image cache')
else:
    if CANON_TEMP.exists():
        assert CANON_STATE.exists(),'Partial canonical cache has no recovery state'
        state=json.loads(CANON_STATE.read_text());assert state['input_fingerprint']==INPUT_ID
        start=int(state['committed']);assert 0<=start<=len(DEV)
        mmap=np.lib.format.open_memmap(CANON_TEMP,mode='r+',dtype=np.uint8)
        assert mmap.shape==SHAPE
        event('RESUME',f'Canonical image cache resumes at image {start}/{len(DEV)}')
    else:
        mmap=np.lib.format.open_memmap(CANON_TEMP,mode='w+',dtype=np.uint8,shape=SHAPE)
        start=0;atomic_json(CANON_STATE,{'input_fingerprint':INPUT_ID,'committed':0})
    progress('canonical image decoding',start,len(DEV))
    with Heartbeat():
        for i in tqdm(range(start,len(DEV)),desc='Canonical RGB cache',initial=start,total=len(DEV)):
            mmap[i]=np.asarray(canonical_image(DEV.iloc[i].verified_image_path),dtype=np.uint8)
            if (i+1)%16==0 or i+1==len(DEV):
                mmap.flush()
                atomic_json(CANON_STATE,{'input_fingerprint':INPUT_ID,'committed':i+1})
                progress('canonical image decoding',i+1,len(DEV))
        del mmap
    os.replace(CANON_TEMP,CANON_FINAL)
    atomic_json(CANON_META,{'input_fingerprint':INPUT_ID,'sha256':file_sha(CANON_FINAL),'shape':SHAPE,'created_utc':utc()})
    CANON_STATE.unlink(missing_ok=True)
    event('CACHE','Canonical image cache complete and SHA-256 pinned')

class StressDataset(Dataset):
    def __init__(self,path,condition):self.path=str(path);self.condition=condition;self._arr=None
    def __len__(self):return len(DEV)
    def __getitem__(self,i):
        if self._arr is None:self._arr=np.load(self.path,mmap_mode='r')
        rgb=Image.fromarray(np.asarray(self._arr[i],dtype=np.uint8),'RGB')
        return PRE(apply_stress(rgb,self.condition))

print('Canonical cache:',CANON_FINAL, 'Size (MiB):',round(CANON_FINAL.stat().st_size/2**20,1))

## 4. Restore and pin original frozen models


In [ ]:
class SizeHead(nn.Module):
    def __init__(self):
        super().__init__();self.layer4=models.resnet18(weights=None).layer4;self.fc=nn.Linear(512,3)
    def forward(self,x):return self.fc(self.layer4(x).mean((-2,-1)))
class FeatureTrunk(nn.Module):
    def __init__(self,resnet):
        super().__init__()
        self.conv1=resnet.conv1;self.bn1=resnet.bn1;self.relu=resnet.relu;self.maxpool=resnet.maxpool
        self.layer1=resnet.layer1;self.layer2=resnet.layer2;self.layer3=resnet.layer3
    def forward(self,x):return self.layer3(self.layer2(self.layer1(self.maxpool(self.relu(self.bn1(self.conv1(x)))))))

models_ready=False;DEVICE=None
if not RUN_INFERENCE:
    event('PREFLIGHT','Inference disabled; caches and development inputs checked, no model evaluation performed')
else:
    available=torch.cuda.is_available()
    if REQUIRE_GPU and not available:
        raise RuntimeError('CUDA GPU currently unavailable. GPU-first mode refuses silent CPU fallback. Restore GPU or explicitly set REQUIRE_GPU=False.')
    DEVICE=torch.device('cuda:0' if available else 'cpu')
    affinity=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else (os.cpu_count() or 1)
    if DEVICE.type=='cuda':
        torch.backends.cudnn.benchmark=True
        torch.backends.cudnn.allow_tf32=True
        torch.backends.cuda.matmul.allow_tf32=True
        try:torch.set_float32_matmul_precision('high')
        except Exception:pass
        name=torch.cuda.get_device_name(0)
        props=torch.cuda.get_device_properties(0)
        event('HARDWARE',f'GPU={name}; VRAM={props.total_memory/2**30:.1f} GiB; CPU affinity={affinity}; CUDA={torch.version.cuda}')
    else:
        torch.set_num_threads(max(1,affinity))
        event('HARDWARE',f'CPU explicitly enabled; CPU affinity={affinity}')
    assert (AUDIT/'EVIDENCE_RECORD.json').is_file()
    assert (TARGETED/'RESULTS_COMPLETE.json').is_file()
    evidence=json.loads((AUDIT/'EVIDENCE_RECORD.json').read_text())
    target=json.loads((TARGETED/'RESULTS_COMPLETE.json').read_text())
    assert evidence['status']=='EXPLORATORY_EVIDENCE_FROZEN'
    assert evidence['development_decision']=='TARGETED_ACCEPTED'
    assert target['development_decision']=='TARGETED_ACCEPTED'
    assert abs(float(evidence['blend_weight'])-0.5)<1e-10
    assert abs(float(target['blend_weight'])-0.5)<1e-10
    selection_path=PILOT/'development_model_selection.json'
    choice=json.loads(selection_path.read_text())
    winner=str(choice['winner']);temperature=float(choice['temperature'])
    assert winner in ('equal_weight_ensemble','seed_11','seed_23','seed_37') and .1<=temperature<=20
    weights=models.ResNet18_Weights.IMAGENET1K_V1
    weights_path=Path(torch.hub.get_dir())/'checkpoints'/Path(urlparse(weights.url).path).name
    assert weights_path.is_file(),'Cached original ResNet-18 ImageNet V1 weights missing; do not silently substitute'
    frozen_paths=[weights_path,selection_path,AUDIT/'EVIDENCE_RECORD.json',TARGETED/'RESULTS_COMPLETE.json',
        RGB/'rgb_representation_finetuned.pt']+[SPATIAL/f'size_layer4_seed{s}.pt' for s in SEEDS]
    frozen_hashes={str(p):file_sha(p) for p in frozen_paths}
    progress('frozen checkpoint restore')
    with Heartbeat():
        pretrained=models.resnet18(weights=None)
        pretrained.load_state_dict(torch.load(weights_path,map_location='cpu',weights_only=True),strict=True)
        trunk=FeatureTrunk(pretrained).eval().to(DEVICE)
        heads={}
        for seed in SEEDS:
            checkpoint=SPATIAL/f'size_layer4_seed{seed}.pt'
            payload=torch.load(checkpoint,map_location='cpu',weights_only=True)
            assert int(payload['config']['seed'])==seed
            m=SizeHead();m.load_state_dict(payload['model_state'],strict=True)
            heads[seed]=m.eval().to(DEVICE)
            event('WEIGHTS',f'Loaded frozen size head seed {seed}')
        rgb_payload=torch.load(RGB/'rgb_representation_finetuned.pt',map_location='cpu',weights_only=True)
        assert rgb_payload['config']['size_seed11_checkpoint_sha256']==frozen_hashes[str(SPATIAL/'size_layer4_seed11.pt')]
        rgb_net=models.resnet18(weights=None);rgb_net.fc=nn.Linear(512,3)
        rgb_net.load_state_dict(rgb_payload['model_state'],strict=True)
        rgb_net=rgb_net.eval().to(DEVICE)
        for model in (trunk,rgb_net,*heads.values()):
            for p in model.parameters():p.requires_grad_(False)
    MODELS_ID=fingerprint({'inputs':INPUT_ID,'version':EXPERIMENT_VERSION,'frozen_hashes':frozen_hashes,
          'winner':winner,'temperature':temperature,'blend':.5,'torch':torch.__version__,'torchvision':__import__('torchvision').__version__})
    event('WEIGHTS',f'All frozen checkpoints loaded, model fingerprint={MODELS_ID[:20]}')
    models_ready=True

## 5. Automatic GPU benchmark and numerical parity guard


In [ ]:
@torch.inference_mode()
def predict_device(x,amp=False):
    ctx=torch.autocast(device_type='cuda',dtype=torch.float16) if amp and DEVICE.type=='cuda' else contextlib.nullcontext()
    with ctx:
        z=trunk(x)
        seed_probs=torch.stack([torch.softmax(heads[s](z),dim=1) for s in SEEDS],dim=0)
        if winner=='equal_weight_ensemble':base=seed_probs.mean(0)
        else:base=seed_probs[SEEDS.index(int(winner.split('_')[1]))]
        base=torch.softmax(torch.log(torch.clamp(base,min=1e-12))/temperature,dim=1)
        cand=base.clone()
        # Only Dandhi needs the RGB candidate, sparing up to ~3/4 of RGB inference.
        # Caller provides a matching boolean tensor for each batch through the shared variable _batch_dandhi.
        if _batch_dandhi.any():
            d_idx=torch.where(_batch_dandhi)[0]
            rp=torch.softmax(rgb_net(x.index_select(0,d_idx))/temperature,dim=1)
            cand[d_idx]=.5*base.index_select(0,d_idx)+.5*rp
    return base.float(),cand.float()

if not models_ready:
    event('BENCH','Skipped (preflight-only)')
else:
    PERF_PLAN=OUT/f'{MODELS_ID[:24]}_hardware_plan.json'
    if PERF_PLAN.exists():
        plan=json.loads(PERF_PLAN.read_text())
        assert plan['model_fingerprint']==MODELS_ID
        EFFECTIVE_BATCH=int(plan['batch_size']);USE_AMP=bool(plan['use_fp16'])
        event('RESUME',f'Reusing performance plan: batch={EFFECTIVE_BATCH}, FP16={USE_AMP}')
    else:
        # Precision parity uses REAL held-in-development images, not artificial random tensors.
        # Choose first four source IDs per variety deterministically; add a blur condition.
        pilot_idx=[int(i) for var in ('Aseel','Dandhi','Karblian','Kupro')
                   for i in DEV.index[DEV.variety.eq(var)].tolist()[:4]]
        canonical_pilot=np.load(CANON_FINAL,mmap_mode='r')
        warm_images=[];pilot_flags=[]
        for i in pilot_idx:
            rgb=Image.fromarray(np.asarray(canonical_pilot[i],dtype=np.uint8),'RGB')
            for stress in ('clean','blur_1_2'):
                warm_images.append(PRE(apply_stress(rgb,stress)))
                pilot_flags.append(DEV.iloc[i].variety=='Dandhi')
        del canonical_pilot
        warm=torch.stack(warm_images).to(DEVICE)
        _batch_dandhi=torch.as_tensor(pilot_flags,device=DEVICE,dtype=torch.bool)
        progress('FP16/FP32 numerical pilot')
        with Heartbeat():
            ref=predict_device(warm,amp=False)
            USE_AMP=False;parity={'fp16_tested':False,'fp16_allowed':False}
            if AUTO_MIXED_PRECISION and DEVICE.type=='cuda':
                try:
                    trial=predict_device(warm,amp=True)
                    diff=max(float((a-b).abs().max().item()) for a,b in zip(ref,trial))
                    same=all(torch.equal(a.argmax(1),b.argmax(1)) for a,b in zip(ref,trial))
                    USE_AMP=bool(same and diff<=AMP_MAX_PROB_DIFF)
                    parity={'fp16_tested':True,'fp16_allowed':USE_AMP,'max_probability_difference':diff,'identical_pilot_predictions':same}
                except (RuntimeError,ValueError) as ex:
                    parity={'fp16_tested':True,'fp16_allowed':False,'error':str(ex)[:200]}
            event('PARITY',parity)
            if BATCH_SIZE=='auto' and DEVICE.type=='cuda':
                candidates=[b for b in (32,64,128,256,384,512) if b<=MAX_AUTO_BATCH]
                scores=[]
                for b in candidates:
                    x=None
                    try:
                        _batch_dandhi=torch.zeros(b,device=DEVICE,dtype=torch.bool);_batch_dandhi[::4]=True
                        x=torch.rand((b,3,224,224),device=DEVICE)
                        torch.cuda.synchronize()
                        t0=time.perf_counter()
                        predict_device(x,amp=USE_AMP)
                        torch.cuda.synchronize()
                        dt=time.perf_counter()-t0
                        scores.append((b,b/max(dt,1e-6)))
                        event('BENCH',f'batch={b} throughput={b/dt:.1f} images/s (synthetic GPU inputs)')
                    except torch.cuda.OutOfMemoryError:
                        event('BENCH',f'batch={b}: out of memory; stop scaling')
                        torch.cuda.empty_cache();break
                    finally:
                        if x is not None: del x
                assert scores, 'No CUDA batch size could be benchmarked'
                best=max(v for b,v in scores)
                EFFECTIVE_BATCH=next(b for b,v in scores if v>=.97*best)
            elif BATCH_SIZE=='auto':
                EFFECTIVE_BATCH=16
            else:
                EFFECTIVE_BATCH=int(BATCH_SIZE)
                assert EFFECTIVE_BATCH>0
            del warm
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        plan={'model_fingerprint':MODELS_ID,'batch_size':EFFECTIVE_BATCH,'use_fp16':USE_AMP,
            'batch_candidates':scores if BATCH_SIZE=='auto' and DEVICE.type=='cuda' else [],
            'parity':parity,'hardware':str(DEVICE),'created_utc':utc()}
        atomic_json(PERF_PLAN,plan)
        event('BENCH',f'Selected batch={EFFECTIVE_BATCH}, FP16={USE_AMP}')
    ncpu=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else (os.cpu_count() or 1)
    WORKERS=min(8,max(0,ncpu-1)) if NUM_WORKERS=='auto' and DEVICE.type=='cuda' else (0 if NUM_WORKERS=='auto' else int(NUM_WORKERS))
    WORKERS=max(0,min(WORKERS,max(0,ncpu-1)))
    # Prevent worker crashes on Jupyter containers with small /dev/shm.
    if WORKERS and Path('/dev/shm').exists():
        shm=os.statvfs('/dev/shm');free_shm=shm.f_bavail*shm.f_frsize
        batch_bytes=EFFECTIVE_BATCH*3*224*224*4
        safe_workers=max(0,int(free_shm//max(1,3*batch_bytes)))
        if safe_workers<WORKERS:
            event('HARDWARE',f'Limiting data workers {WORKERS}->{safe_workers}: free /dev/shm={free_shm/2**20:.0f}MiB, batch tensors={batch_bytes/2**20:.0f}MiB')
            WORKERS=safe_workers
    event('HARDWARE',f'DataLoader workers={WORKERS}; pinned host transfers={DEVICE.type=="cuda"}; batch={EFFECTIVE_BATCH}')

## 6. Recorded inference with checkpointed **minibatch** cache


In [ ]:
def atomic_npz(path,**arrays):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with tmp.open('wb') as f:
        np.savez_compressed(f,**arrays);f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def block_path(kind,idx):return OUT/'prediction_cache'/MODELS_ID[:24]/kind/f'block_{idx:05d}.npz'
def load_block(kind,idx,start,end):
    p=block_path(kind,idx)
    if not p.exists():return None
    try:
        with np.load(p,allow_pickle=False) as d:
            assert str(d['model_fingerprint'].item())==MODELS_ID
            assert str(d['condition'].item())==kind
            assert int(d['start'].item())==start and int(d['end'].item())==end
            assert np.array_equal(d['source_ids'],DEV.source_id.iloc[start:end].astype(str).to_numpy())
            base=np.array(d['baseline'],dtype=np.float32)
            cand=np.array(d['candidate'],dtype=np.float32)
        assert base.shape==cand.shape==(end-start,3)
        expected_digest=hashlib.sha256((MODELS_ID+'|'+kind+'|'+str(start)+'|'+str(end)).encode()+
                np.ascontiguousarray(base).tobytes()+np.ascontiguousarray(cand).tobytes()).hexdigest()
        with np.load(p,allow_pickle=False) as d:
            assert str(d['payload_sha256'].item())==expected_digest,'Cached numeric payload checksum mismatch'
        assert np.isfinite(base).all() and np.isfinite(cand).all()
        assert np.allclose(base.sum(1),1,atol=3e-3) and np.allclose(cand.sum(1),1,atol=3e-3)
        return base,cand
    except Exception as ex:
        event('CACHE',f'Invalid cached block {p.name} ({kind}): {ex}. It will be recomputed.')
        return None

if not models_ready:
    event('INFERENCE','Not executed (preflight-only)')
else:
    INFER_PLAN=OUT/'inference_plan.json'
    if INFER_PLAN.exists():
        cp=json.loads(INFER_PLAN.read_text())
        assert cp['model_fingerprint']==MODELS_ID
        # Resume previous chunks even if the auto batch benchmark is different on restart.
        CHUNK_ROWS=int(cp['chunk_rows'])
    else:
        CHUNK_ROWS=EFFECTIVE_BATCH
        atomic_json(INFER_PLAN,{'model_fingerprint':MODELS_ID,'chunk_rows':CHUNK_ROWS,
                    'conditions':CONDITIONS,'source_count':len(DEV),'created_utc':utc()})
    chunks=[(i,min(i+CHUNK_ROWS,len(DEV))) for i in range(0,len(DEV),CHUNK_ROWS)]
    total_blocks=len(chunks)*len(CONDITIONS)
    n_cached=sum(load_block(c,j,s,e) is not None for c in CONDITIONS for j,(s,e) in enumerate(chunks))
    progress('inference',n_cached,total_blocks)
    event('INFERENCE',f'Found {n_cached}/{total_blocks} verified cached blocks. Running missing blocks only.')
    with Heartbeat():
        with tqdm(total=total_blocks,initial=n_cached,desc='All frozen stress batches',unit='batch') as bar:
            for kind in CONDITIONS:
                missing=[j for j,(s,e) in enumerate(chunks) if load_block(kind,j,s,e) is None]
                if not missing:
                    event('CACHE',f'Condition {kind} fully reused');continue
                ds=StressDataset(CANON_FINAL,kind)
                sampler=[list(range(chunks[j][0],chunks[j][1])) for j in missing]
                loader=DataLoader(ds,batch_sampler=sampler,num_workers=WORKERS,
                        pin_memory=(DEVICE.type=='cuda'),persistent_workers=(WORKERS>0),
                        prefetch_factor=2 if WORKERS>0 else None)
                event('INFERENCE',f'{kind}: {len(missing)} remaining batches; {sum(chunks[j][1]-chunks[j][0] for j in missing)} images')
                for j,x in zip(missing,loader):
                    start,end=chunks[j];assert x.shape[0]==end-start
                    xs=x.to(DEVICE,non_blocking=(DEVICE.type=='cuda'))
                    global _batch_dandhi
                    _batch_dandhi=torch.as_tensor(DEV.variety.iloc[start:end].eq('Dandhi').to_numpy(),device=DEVICE,dtype=torch.bool)
                    with torch.inference_mode():
                        b,c=predict_device(xs,amp=USE_AMP)
                        if DEVICE.type=='cuda':torch.cuda.synchronize()
                        b=b.cpu().numpy();c=c.cpu().numpy()
                    assert np.array_equal(b[~DEV.variety.iloc[start:end].eq('Dandhi').to_numpy()],
                                          c[~DEV.variety.iloc[start:end].eq('Dandhi').to_numpy()]),'Non-Dandhi candidate differs'
                    _b=b.astype(np.float32);_c=c.astype(np.float32)
                    _digest=hashlib.sha256((MODELS_ID+'|'+kind+'|'+str(start)+'|'+str(end)).encode()+
                        np.ascontiguousarray(_b).tobytes()+np.ascontiguousarray(_c).tobytes()).hexdigest()
                    atomic_npz(block_path(kind,j),model_fingerprint=np.array(MODELS_ID),condition=np.array(kind),
                      start=np.array(start),end=np.array(end),source_ids=np.asarray(DEV.source_id.iloc[start:end].astype(str).tolist(),dtype=np.str_),
                      baseline=_b,candidate=_c,payload_sha256=np.array(_digest))
                    n_cached+=1;progress(f'inference {kind}',n_cached,total_blocks)
                    bar.update(1)
                    if DEVICE.type=='cuda':
                        mem=f' GPU peak allocated={torch.cuda.max_memory_allocated()/2**30:.2f}GiB'
                    else:mem=''
                    event('BATCH',f'{kind} block {j+1}/{len(chunks)} source rows {start+1}-{end}; saved{mem}')
                del loader
                event('INFERENCE',f'Condition {kind} completed')
    event('INFERENCE',f'All {total_blocks} blocks stored. Metrics may be regenerated without model inference.')

## 7. Rebuild complete outputs from verified cached predictions (idempotent)


In [ ]:
if not models_ready:
    print('Preflight-only: no inference output was generated.')
else:
    progress('report reconstruction')
    outputs=[]
    for kind in tqdm(CONDITIONS,desc='Reconstructing reports'):
        pieces=[load_block(kind,j,s,e) for j,(s,e) in enumerate(chunks)]
        assert all(x is not None for x in pieces),f'Missing block(s) for {kind}; rerun inference cell'
        bp=np.concatenate([x[0] for x in pieces]);cp=np.concatenate([x[1] for x in pieces])
        assert bp.shape==cp.shape==(len(DEV),3)
        for model,probs in [('baseline',bp),('Dandhi_targeted',cp)]:
            frame=DEV[['source_id','source_group','variety','native_size_label']].copy()
            frame['condition']=kind;frame['model']=model
            frame['pred_size']=np.asarray(SIZES)[probs.argmax(1)]
            frame['max_probability']=probs.max(1)
            frame['correct']=frame.pred_size.eq(frame.native_size_label)
            outputs.append(frame)
    pred=pd.concat(outputs,ignore_index=True)
    assert len(pred)==len(DEV)*len(CONDITIONS)*2
    def atomic_csv(frame,path):
        tmp=Path(str(path)+'.tmp');frame.to_csv(tmp,index=False);os.replace(tmp,path)
    atomic_csv(pred,OUT/'stress_test_per_image_predictions.csv')
    metrics=pred.groupby(['condition','model','variety'],as_index=False).agg(
      n=('correct','size'),accuracy=('correct','mean'),mean_max_probability=('max_probability','mean'))
    atomic_csv(metrics,OUT/'stress_test_variety_summary.csv')
    ref=pred[pred.condition=='clean'][['source_id','model','pred_size','correct']].rename(
        columns={'pred_size':'clean_pred','correct':'clean_correct'})
    paired=pred.merge(ref,on=['source_id','model'],validate='many_to_one')
    paired['grade_changed']=paired.pred_size.ne(paired.clean_pred)
    group=paired.groupby(['condition','model','variety']).agg(n=('source_id','size'),
        accuracy=('correct','mean'),clean_accuracy=('clean_correct','mean'),
        grade_change=('grade_changed','mean')).reset_index()
    group['accuracy_delta']=group.accuracy-group.clean_accuracy
    atomic_csv(group,OUT/'stress_test_paired_summary.csv')
    summary=group.pivot_table(index=['condition','variety'],columns='model',values=['accuracy','accuracy_delta','grade_change']).round(4)
    manifest={'status':'EXPLORATORY_DEVELOPMENT_STRESS_COMPLETE','created_utc':utc(),
      'run_version':EXPERIMENT_VERSION,'model_fingerprint':MODELS_ID,'input_fingerprint':INPUT_ID,
      'conditions':list(CONDITIONS),'n_original_sources':len(DEV),'n_prediction_rows':len(pred),
      'frozen_checkpoint_sha256':frozen_hashes,'frozen_temperature':temperature,
      'frozen_selection':winner,'frozen_Dandhi_blend_weight':.5,
      'batch_rows':CHUNK_ROWS,'hardware_plan':plan,'cpu_workers':WORKERS,'device':str(DEVICE),
      'external_validation':False,'new_training':False,'locked_test_accessed':False,
      'reports':{name:file_sha(OUT/name) for name in ['stress_test_per_image_predictions.csv','stress_test_variety_summary.csv','stress_test_paired_summary.csv']}}
    atomic_json(OUT/'STRESS_PROVENANCE.json',manifest)
    event('COMPLETE',f'{len(pred)} saved predictions, {len(CONDITIONS)} conditions, {len(DEV)} distinct development originals')
    print('\nPaired stress deterioration by variety and frozen grader:')
    display(summary)
    print('\nReports:',OUT)

## Outputs and interpretation
